# ZeroHunter — تدريب النموذج وتصدير GGUF

تأكد من اختيار **Runtime → Change runtime type → T4 GPU** قبل تشغيل الخلايا.

In [ ]:
%cd /content
!apt-get update -qq && apt-get install -y -qq cmake libcurl4-openssl-dev build-essential > /dev/null
!pip install -q --no-cache-dir unsloth gguf protobuf sentencepiece
!rm -rf zh_repo && git clone --depth 1 https://github.com/sdxdxa56-design/jdjsjsh.git zh_repo
%cd /content/zh_repo

In [ ]:
# تحديث بيانات الإنترنت قبل البدء وتدريب النموذج مع دمج ملفات التجربة
!python3 -c "from core.knowledge import build_curated_dataset; build_curated_dataset()" || true
!python3 train.py

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
except Exception:
    pass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("أدخل HF_TOKEN لرفع النموذج إلى Hugging Face: ")

!python3 upload_hf.py

### خلية 4 (اختيارية): تشغيل خادم الدردشة السريع (1–4 ثوانٍ للرد) وربطه عبر ZEROHUNTER_LOCAL_URL

شغّل هذه الخلية لتحويل جلسة Colab إلى خادم استدلال سريع، ثم انسخ الرابط الظاهر (`https://....trycloudflare.com`) وضعه في AI Studio Secrets باسم `ZEROHUNTER_LOCAL_URL`.

In [ ]:
!pip install -q flask requests huggingface_hub llama-cpp-python
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

import os
import re
import time
import subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download

gguf_candidates = sorted(Path(".").rglob("*.gguf"), key=lambda p: p.stat().st_size, reverse=True)
if gguf_candidates:
    model_path = str(gguf_candidates[0].resolve())
else:
    model_path = hf_hub_download(
        repo_id="Qwen/Qwen2.5-Coder-3B-Instruct-GGUF",
        filename="qwen2.5-coder-3b-instruct-q4_k_m.gguf",
    )

env = os.environ.copy()
env["ZEROHUNTER_LOCAL_MODEL"] = model_path
with open("web_app.log", "w") as web_out:
    subprocess.Popen(["python3", "web_app.py"], env=env, stdout=web_out, stderr=subprocess.STDOUT)

with open("cf.log", "w") as cf_out:
    subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:5000", "--no-autoupdate"],
        stdout=cf_out,
        stderr=subprocess.STDOUT,
    )

public_url = ""
for _ in range(45):
    time.sleep(1)
    if Path("cf.log").exists():
        log_txt = Path("cf.log").read_text(encoding="utf-8", errors="ignore")
        match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", log_txt)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print("خادم ZeroHunter السريع يعمل الآن!")
    print("انسخ هذا الرابط وضعه في Secrets باسم ZEROHUNTER_LOCAL_URL:")
    print(public_url)
else:
    print("تعذر التقاط رابط النفق تلقائياً. راجع cf.log و web_app.log.")